In [26]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

In [12]:
df = pd.read_csv('/content/dispute_dataset.csv')

In [13]:
df.shape

(6000, 15)

In [14]:
df.head()

,case_id,reason_code,product_category,payment_method,order_value,time_to_dispute_days,days_remaining_to_respond,delivery_confirmed,communication_logs_present,device_ip_match_score,listing_accuracy_score,customer_prior_disputes,customer_account_age_days,_true_win_prob_debug_only,merchant_won
0,90b53ba6-0656-4acd-9f0d-ddf9472e60ea,not_as_described,groceries,netbanking,2093.32,1,10,not_shipped,False,0.620,0.536,0,41,0.2470,False
1,97f75f3a-0204-4401-9077-6d47ffadaa65,not_as_described,electronics,card,2698.06,13,16,in_transit,True,0.521,0.425,0,133,0.3349,False
2,b56931ac-6222-4be0-9cec-d202f72f785b,item_not_received,electronics,upi,1463.87,12,18,in_transit,True,0.499,0.515,1,164,0.2949,True
3,2f6cc00b-2c4a-4962-a24c-87c7806a7db6,not_as_described,groceries,upi,664.21,22,10,delivered_no_signature,False,0.719,0.800,0,1026,0.7064,True
4,144bead8-545f-4fed-8716-f196d772ad68,duplicate_charge,beauty,card,1354.38,2,14,signed_confirmation,True,0.327,0.568,0,217,0.4357,True


In [15]:
df.tail()

,case_id,reason_code,product_category,payment_method,order_value,time_to_dispute_days,days_remaining_to_respond,delivery_confirmed,communication_logs_present,device_ip_match_score,listing_accuracy_score,customer_prior_disputes,customer_account_age_days,_true_win_prob_debug_only,merchant_won
5995,66edf122-2d14-4433-8328-b4ac20fc9924,item_not_received,beauty,upi,211.99,12,15,delivered_no_signature,False,0.741,0.394,1,255,0.5597,True
5996,a9329a76-5b51-4840-b046-dee94f3c1691,item_not_received,electronics,wallet,984.35,3,16,signed_confirmation,False,0.631,0.331,0,2355,0.6570,False
5997,093ffc0f-0361-47a6-a403-1ea61baa3149,item_not_received,groceries,card,1968.84,4,6,signed_confirmation,False,0.915,0.640,0,545,0.8019,True
5998,6b6e6818-f76a-4a3d-b5f9-0b7df7bdb005,fraud_card_not_present,home_goods,upi,1075.58,9,11,signed_confirmation,False,0.488,0.444,1,8,0.4461,True
5999,ac1e3c4b-2c4d-421d-a491-feed261ed8e4,item_not_received,apparel,card,981.84,2,10,delivered_no_signature,True,0.743,0.213,1,142,0.5811,True


In [16]:
df = df.drop(columns=['_true_win_prob_debug_only','case_id'],axis=1)

In [17]:
df.shape

(6000, 13)

In [18]:
df.head()

,reason_code,product_category,payment_method,order_value,time_to_dispute_days,days_remaining_to_respond,delivery_confirmed,communication_logs_present,device_ip_match_score,listing_accuracy_score,customer_prior_disputes,customer_account_age_days,merchant_won
0,not_as_described,groceries,netbanking,2093.32,1,10,not_shipped,False,0.620,0.536,0,41,False
1,not_as_described,electronics,card,2698.06,13,16,in_transit,True,0.521,0.425,0,133,False
2,item_not_received,electronics,upi,1463.87,12,18,in_transit,True,0.499,0.515,1,164,True
3,not_as_described,groceries,upi,664.21,22,10,delivered_no_signature,False,0.719,0.800,0,1026,True
4,duplicate_charge,beauty,card,1354.38,2,14,signed_confirmation,True,0.327,0.568,0,217,True


In [19]:
df.describe()

,order_value,time_to_dispute_days,days_remaining_to_respond,device_ip_match_score,listing_accuracy_score,customer_prior_disputes,customer_account_age_days
count,6000.000000,6000.000000,6000.000000,6000.000000,6000.000000,6000.000000,6000.000000
mean,1343.652465,10.667667,11.034667,0.503004,0.600633,0.603167,396.926833
std,1550.067924,10.218657,6.057358,0.225534,0.198623,0.771442,387.187066
min,35.640000,1.000000,1.000000,0.005000,0.037000,0.000000,5.000000
25%,489.752500,3.000000,6.000000,0.326000,0.457000,0.000000,118.000000
50%,876.635000,7.000000,11.000000,0.507000,0.614000,0.000000,285.000000
75%,1619.657500,15.000000,16.000000,0.679000,0.761000,1.000000,553.000000
max,25550.880000,93.000000,21.000000,0.994000,0.993000,5.000000,3454.000000


In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   reason_code                 6000 non-null   object 
 1   product_category            6000 non-null   object 
 2   payment_method              6000 non-null   object 
 3   order_value                 6000 non-null   float64
 4   time_to_dispute_days        6000 non-null   int64  
 5   days_remaining_to_respond   6000 non-null   int64  
 6   delivery_confirmed          6000 non-null   object 
 7   communication_logs_present  6000 non-null   bool   
 8   device_ip_match_score       6000 non-null   float64
 9   listing_accuracy_score      6000 non-null   float64
 10  customer_prior_disputes     6000 non-null   int64  
 11  customer_account_age_days   6000 non-null   int64  
 12  merchant_won                6000 non-null   bool   
dtypes: bool(2), float64(3), int64(4),

In [21]:
df['reason_code'].value_counts()

,count
reason_code,
item_not_received,1804
fraud_card_not_present,1661
not_as_described,1194
duplicate_charge,735
subscription_not_cancelled,606


In [22]:
df['product_category'].value_counts()

,count
product_category,
digital_goods,1037
apparel,1023
groceries,1021
electronics,1005
home_goods,990
beauty,924


In [23]:
df['merchant_won'].value_counts()

,count
merchant_won,
True,3322
False,2678


Label Encoding

In [27]:
cat_cols = ['reason_code','product_category','payment_method','delivery_confirmed']

In [28]:
df_enc = pd.get_dummies(df,columns=cat_cols,drop_first=False)

In [29]:
X = df_enc.drop(columns=['merchant_won'])
y = df_enc["merchant_won"].astype(int)

In [30]:
X.shape

(6000, 27)

Model Training

In [31]:
X_train,X_temp,y_train,y_temp = train_test_split(
    X,y,test_size=0.4,stratify=y,random_state=42
    )

In [32]:
X_cal,X_test,y_cal,y_test = train_test_split(
    X_temp,y_temp,test_size=0.5,stratify=y_temp,random_state=42
)

In [33]:
print(f"Train: {X_train.shape[0]}  Calibration: {X_cal.shape[0]}  Test: {X_test.shape[0]}")

Train: 3600  Calibration: 1200  Test: 1200


In [34]:
raw_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.08,
    eval_metric='logloss',
    random_state=42
)

In [35]:
raw_model.fit(X_train,y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.08, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [36]:
raw_probs_test = raw_model.predict_proba(X_test)[:, 1]

In [37]:
raw_probs_test[:10]

array([0.7595633 , 0.5856308 , 0.49456024, 0.36225486, 0.6398554 ,
       0.61658555, 0.37198886, 0.7029528 , 0.3544439 , 0.2673111 ],
      dtype=float32)